# 01：文本 LoRA 环境检查

先按 `text-lora/README.md` 手动安装依赖并注册 kernel，再选择 **Python (Text LoRA - Qwen)**。

按顺序运行全部 Cell。这份 Notebook 复用命令行检查脚本，只检查环境和本地模型配置/tokenizer，不安装包、不下载模型、不加载模型权重、不开始训练。

In [2]:
import sys
from pathlib import Path

# 兼容从仓库根目录、text-lora/ 或 notebooks/ 启动 Jupyter。
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path
    for candidate in candidates
    for path in (candidate, candidate / "text-lora")
    if (path / "scripts" / "check_environment.py").is_file()
    and (path / "configs" / "models.toml").is_file()
)
sys.path.insert(0, str(project_root / "scripts"))

print("项目目录：", project_root)

项目目录： /home/muyi086/github/test_LLM/text-lora


In [3]:
from check_environment import (
    check_gpu,
    check_local_model,
    check_packages,
    check_python,
    check_tokenizer,
    load_model_config,
    run_check,
)

# None 使用 models.toml 的默认模型；备用配置为 'qwen3_4b_instruct'。
MODEL_PROFILE = None
model = load_model_config(MODEL_PROFILE)
print("本地模型：", model.name, model.path)

本地模型： Qwen3.5-4B /home/muyi086/hf-mirror/Qwen/Qwen3.5-4B


## Python / kernel

应使用 Python 3.12，`sys.prefix` 应指向 `text-lora/.venv`。如果失败，先切换 kernel。

In [4]:
assert run_check("Python / kernel", check_python), "请切换为项目 kernel"


--- Python / kernel ---
Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3
[PASS]


## 训练依赖

列出已安装的实际版本并检查能否导入。缺失包及兼容性错误会集中显示。

In [5]:
assert run_check("训练依赖", check_packages), "请按 README 手动处理依赖"


--- 训练依赖 ---
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
ms-swift: 4.5.3
transformers: 5.16.1
peft: 0.19.1
datasets: 4.8.4
accelerate: 1.15.0
modelscope: 1.40.1
qwen-vl-utils: 0.0.14
ipykernel: 7.4.0
[PASS]


## CUDA / GPU / BF16

CUDA 和 BF16 必须可用。空闲显存是当前快照，不代表后续训练的峰值显存。

In [6]:
assert run_check("GPU / BF16", check_gpu), "请检查项目的 CUDA wheel 和 WSL 驱动"


--- GPU / BF16 ---
PyTorch：2.11.0+cu130
CUDA runtime：13.0
GPU：NVIDIA GeForce RTX 4070 Ti SUPER
BF16：True
显存：空闲 14.73 / 总量 15.99 GiB
[PASS]


## 本地模型文件

模型来自 `configs/models.toml`。检查架构、聊天模板和索引引用的全部权重分片是否存在且非空。

In [9]:
assert run_check("本地模型文件", lambda: check_local_model(model)), "请检查本地模型目录"


--- 本地模型文件 ---
模型：Qwen3.5-4B
路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
架构：['Qwen3_5ForConditionalGeneration']
权重：2 个文件，8.68 GiB
检查了文件存在性和非空状态；未进行权重校验和验证。
[PASS]


## 离线架构 / tokenizer

确认 Transformers 能识别模型架构，并使用本地 tokenizer 渲染文本聊天模板。不会加载模型权重。

In [7]:
assert run_check("离线架构 / tokenizer", lambda: check_tokenizer(model)), (
    "请检查 Transformers 版本与 tokenizer"
)
print("第一阶段检查通过。下一步固定基准提示词、准备教学数据，再添加训练脚本。")


--- 离线架构 / tokenizer ---
Transformers 已识别：qwen3_5
Tokenizer：Qwen2Tokenizer
聊天模板可用：81 个字符；未加载模型权重。
[PASS]
第一阶段检查通过。下一步固定基准提示词、准备教学数据，再添加训练脚本。
